# Tutorial 8: Eye Tracking on Aria Gen 2

## Introduction

Aria Gen 2 can produce eye gaze from **three different sources**:

| Source | Where it runs | Available | This tutorial |
| :-- | :-- | :-- | :-- |
| On-device **geometric ET** | On the glasses, in real time | As soon as the recording exists | **Part A** |
| On-device **ML ET** | On the glasses, in real time | As soon as the recording exists | **Part B** |
| **MPS ML ET** | Machine Perception Services, in the cloud | After you upload and MPS finishes | **Part C** |

All three return the **same `EyeGaze` structure**, so geometry and analysis code written against one
source works unchanged against the others.

The two on-device sources are written to the same VRS `eyegaze` stream (`373-1`). A single recording
contains **one** of them, never both — which one you get is decided by the profile used at recording
or streaming time.

**How to read this tutorial**

Parts A, B and C are **independent and self-contained**. If you already know which source your data
came from, jump straight to that part and read it start to finish — you will not need to backtrack.
The parts deliberately repeat material where it applies to more than one source.

**What you'll learn:**

- How to load eye gaze from each of the three sources
- What every field of the `EyeGaze` structure means, and which ones are reachable from Python
- The gaze-geometry helper functions, and which one is correct for which source
- How to project gaze into a camera image and visualize it with Rerun

**Prerequisites**
- Complete Tutorial 1 (VrsDataProvider Basics) to understand basic data provider concepts
- Complete Tutorial 2 (Device Calibration) to understand how to properly use calibration in Aria data
- For on-device hand tracking see Tutorial 4; for the rest of the MPS outputs see Tutorial 7

### ⚠️ Important Notes
- **Google Colab Users:**  
  If you encounter a `ModuleNotFoundError: No module named 'rerun'` error after installing `rerun-sdk`, Colab may not recognize the new package until the runtime is restarted.  
  **Fix:** Go to **Runtime → Restart session and run all**.

- **Visualization Issue :**  
  If a Rerun visualization window does not appear, this may be due to a known caching issue. Simply re-run the visualization cell to resolve it.

## Setup Environment (Google Colab)

If running on Google Colab, install `projectaria-tools` and download the sample data.

This tutorial uses up to four inputs. **Only the first is required.** Every part checks for its own
input first and skips cleanly with a message when it is missing, so the notebook runs end to end no
matter how many of them you have.

| Variable | Contents | Needed by |
| :-- | :-- | :-- |
| `geometric_vrs_file_path` | A VRS whose `eyegaze` stream came from the geometric algorithm | Part A |
| `ml_et_vrs_file_path` | A VRS whose `eyegaze` stream came from ML ET | Part B |
| `mps_folder_path` | An MPS output folder containing an `eye_gaze/` subfolder | Part C |
| `mps_vrs_file_path` | The VRS that `mps_folder_path` was produced from | Part C visualization only |

In [ ]:
import os
import sys

google_colab_env = 'google.colab' in str(get_ipython())

if google_colab_env:
    print("Running from Google Colab, installing projectaria_tools and downloading sample data")

    # Install projectaria-tools
    !pip install projectaria-tools==2.3.0

    # Set up data path
    vrs_sample_path = "./vrs_sample_data"

    # Sample VRS file URL
    vrs_url = "https://www.projectaria.com/async/sample/download/?bucket=core&filename=aria_gen2_sample_data_1.vrs"
    vrs_filename = "aria_gen2_sample_data_1.vrs"
    geometric_vrs_file_path = os.path.join(vrs_sample_path, vrs_filename)

    # Download commands
    command_list = [
        f"mkdir -p {vrs_sample_path}",
        f'curl -o {geometric_vrs_file_path} -C - -O -L "{vrs_url}"',
    ]

    # Execute the commands for downloading dataset
    print(f"Downloading VRS sample data to {geometric_vrs_file_path}...")
    for command in command_list:
        !$command

    print(f"Download complete! VRS file saved to: {geometric_vrs_file_path}")

    # Running this command to trigger early failure of importing ReRun.
    # Should be resolved by restarting the Colab session.
    import rerun as rr
else:
    # For local environment, user needs to specify their own VRS file path
    geometric_vrs_file_path = "path/to/your/geometric_et_recording.vrs"
    print("Please update geometric_vrs_file_path to point to your VRS file")

# Optional inputs. Leave as None to skip the corresponding part.
ml_et_vrs_file_path = None  # e.g. "path/to/your/ml_et_recording.vrs"
mps_folder_path = None  # e.g. "path/to/your/mps_output/"
mps_vrs_file_path = None  # the VRS that mps_folder_path was produced from

### Helpers shared by all three parts

Four small utilities that are the same regardless of where the gaze came from. Everything
source-specific stays inside its own part.

One of them is worth calling out. `find_eyegaze_stream` checks **both** that the stream label
resolves **and** that the stream actually holds samples, because on Gen 2 those are separate
questions: `get_stream_id_from_label` answers from a mostly static device-model table and returns
an id for every label the glasses *could* record, so it is not on its own a presence check.

In [ ]:
from datetime import timedelta

import numpy as np

from projectaria_tools.core.sensor_data import TimeDomain, TimeQueryOptions

EYEGAZE_LABEL = "eyegaze"
RGB_CAMERA_LABEL = "camera-rgb"


def timestamp_to_ns(tracking_timestamp):
    """
    EyeGaze.tracking_timestamp is a datetime.timedelta with microsecond resolution.
    Convert it to integer nanoseconds in the device time domain.
    """
    return (tracking_timestamp // timedelta(microseconds=1)) * 1000


def print_eye_gaze_sample(eye_gaze, title):
    """Print every EyeGaze field that is reachable from Python."""
    print(f"=== {title} ===")
    print(f"  tracking_timestamp:       {timestamp_to_ns(eye_gaze.tracking_timestamp)} ns")
    print(f"  session_uid:              '{eye_gaze.session_uid}'")

    print("  -- combined gaze --")
    print(f"  yaw / pitch:              {eye_gaze.yaw:.4f} / {eye_gaze.pitch:.4f} rad")
    print(f"  yaw   [low, high]:        [{eye_gaze.yaw_low:.4f}, {eye_gaze.yaw_high:.4f}] rad")
    print(f"  pitch [low, high]:        [{eye_gaze.pitch_low:.4f}, {eye_gaze.pitch_high:.4f}] rad")
    print(f"  depth:                    {eye_gaze.depth:.4f} m  (0 means unavailable)")
    print(f"  combined_gaze_valid:      {eye_gaze.combined_gaze_valid}")
    if eye_gaze.combined_gaze_valid:
        print(f"  combined_gaze_origin_in_cpf:  {eye_gaze.combined_gaze_origin_in_cpf}")
    print(f"  spatial_gaze_point_valid: {eye_gaze.spatial_gaze_point_valid}")
    if eye_gaze.spatial_gaze_point_valid:
        print(f"  spatial_gaze_point_in_cpf:    {eye_gaze.spatial_gaze_point_in_cpf}")

    vergence = eye_gaze.vergence
    print("  -- per eye --")
    print(f"  left  yaw / pitch:        {vergence.left_yaw:.4f} / {vergence.left_pitch:.4f} rad")
    print(f"  right yaw / pitch:        {vergence.right_yaw:.4f} / {vergence.right_pitch:.4f} rad")
    print(f"  left  yaw [low, high]:    [{vergence.left_yaw_low:.4f}, {vergence.left_yaw_high:.4f}] rad")
    print(f"  right yaw [low, high]:    [{vergence.right_yaw_low:.4f}, {vergence.right_yaw_high:.4f}] rad")
    print(f"  left  eye origin (CPF):   [{vergence.tx_left_eye:.4f}, {vergence.ty_left_eye:.4f}, {vergence.tz_left_eye:.4f}]")
    print(f"  right eye origin (CPF):   [{vergence.tx_right_eye:.4f}, {vergence.ty_right_eye:.4f}, {vergence.tz_right_eye:.4f}]")
    print(f"  blink  left / right:      {vergence.left_blink} / {vergence.right_blink}")
    print(f"  gaze valid  left / right: {vergence.left_gaze_valid} / {vergence.right_gaze_valid}")
    print(f"  blink valid left / right: {vergence.left_blink_valid} / {vergence.right_blink_valid}")
    print()


def find_eyegaze_stream(provider, vrs_path):
    """
    Return (stream_id, num_samples) for the eyegaze stream, or raise.

    Two things have to be checked, not one. On Gen 2 `get_stream_id_from_label`
    answers from a mostly static device-model table, so it hands back an id for
    every label the glasses *could* record, whether or not this particular file
    carries that stream. A stream can also be declared with zero data records.
    Either case would sail past a plain `is None` check and then fail on the
    first indexed read.
    """
    stream_id = provider.get_stream_id_from_label(EYEGAZE_LABEL)
    if stream_id is None:
        raise RuntimeError(f"'{EYEGAZE_LABEL}' is not a known stream label for {vrs_path}.")

    num_samples = provider.get_num_data(stream_id)
    if num_samples == 0:
        raise RuntimeError(
            f"'{EYEGAZE_LABEL}' stream is declared in {vrs_path} but carries no samples. "
            "Please use a recording made with eye tracking enabled."
        )
    return stream_id, num_samples


def gaze_point_to_pixel(spatial_gaze_point_in_cpf, camera_calib, T_device_cpf):
    """
    Project a CPF-frame gaze point into a camera image.

    Returns a pixel, or None when the point falls outside the camera's valid
    projection area.

    Both legs stay on the calibrated device frame. `get_transform_device_cpf()`
    defaults to the SVD CPF on Gen 2, which is fitted against the *calibrated*
    camera positions, so the camera leg has to be the calibrated
    `get_transform_device_camera()` too. Substituting the CAD extrinsics
    (`get_transform_device_sensor(label, get_cad_value=True)`) mixes the two
    frames -- `get_transform_cpf_sensor` raises on that exact combination -- and
    silently shifts the marker by a degree or so.
    """
    point_in_device = T_device_cpf @ spatial_gaze_point_in_cpf
    point_in_camera = camera_calib.get_transform_device_camera().inverse() @ point_in_device
    return camera_calib.project(point_in_camera)


def overlay_gaze_on_rgb(
    provider,
    gaze_sources,
    rerun_app_name,
    duration_sec=10,
):
    """
    Overlay one or more gaze sources on the first `duration_sec` of RGB frames.

    `gaze_sources` is a list of `(name, gaze_at_time_ns, color)`. Each
    `gaze_at_time_ns` is a callable taking a device-time timestamp in nanoseconds
    and returning an EyeGaze (or None), so this helper never needs to know where
    the gaze came from. Sources are drawn as separate entities, which is what
    makes two of them comparable on the same frame.
    """
    import rerun as rr

    rgb_stream_id = provider.get_stream_id_from_label(RGB_CAMERA_LABEL)
    if rgb_stream_id is None or provider.get_num_data(rgb_stream_id) == 0:
        print(f"This recording has no {RGB_CAMERA_LABEL} frames - nothing to overlay onto.")
        return

    device_calib = provider.get_device_calibration()
    T_device_cpf = device_calib.get_transform_device_cpf()
    rgb_camera_calib = device_calib.get_camera_calib(RGB_CAMERA_LABEL)

    rr.init(rerun_app_name)

    # Deliver only RGB images; gaze is fetched per frame by timestamp so that it is
    # sampled at exactly the frame time.
    deliver_options = provider.get_default_deliver_queued_options()
    deliver_options.deactivate_stream_all()
    deliver_options.activate_stream(rgb_stream_id)

    total_length_ns = provider.get_last_time_ns_all_streams(
        TimeDomain.DEVICE_TIME
    ) - provider.get_first_time_ns_all_streams(TimeDomain.DEVICE_TIME)

    # Start at the beginning of the recording, so no first-side truncation is
    # needed, and cap the window at duration_sec. Capping against the recording
    # length keeps a clip shorter than duration_sec from truncating past its own
    # end, which would leave the viewer empty with no indication why.
    duration_ns = min(int(duration_sec * 1e9), total_length_ns)
    deliver_options.set_truncate_last_device_time_ns(max(total_length_ns - duration_ns, 0))
    print(
        f"  window: first {duration_ns / 1e9:.1f}s of a {total_length_ns / 1e9:.1f}s recording"
    )

    for sensor_data in provider.deliver_queued_sensor_data(deliver_options):
        device_time_ns = sensor_data.get_time_ns(TimeDomain.DEVICE_TIME)
        image_data_and_record = sensor_data.image_data_and_record()

        rr.set_time("device_time", duration=np.timedelta64(device_time_ns, "ns"))
        # A full-resolution Gen 2 RGB frame decodes to ~15 MB, so a 10 s window is
        # ~1.4 GB uncompressed. The notebook viewer runs in wasm with a 1 GiB budget
        # and silently drops the oldest data past it, so the start of the clip loses
        # its images. JPEG keeps the same window roughly 25x smaller.
        rr.log(
            RGB_CAMERA_LABEL,
            rr.Image(image_data_and_record[0].to_numpy_array()).compress(
                jpeg_quality=90
            ),
        )

        for name, gaze_at_time_ns, color in gaze_sources:
            eye_gaze = gaze_at_time_ns(device_time_ns)
            pixel = None
            if eye_gaze is not None and eye_gaze.spatial_gaze_point_valid:
                pixel = gaze_point_to_pixel(
                    eye_gaze.spatial_gaze_point_in_cpf, rgb_camera_calib, T_device_cpf
                )

            entity = f"{RGB_CAMERA_LABEL}/gaze/{name}"
            if pixel is not None:
                rr.log(
                    entity,
                    rr.Points2D(positions=[pixel], colors=[color], radii=[30.0]),
                )
            else:
                # Rerun entities persist along the timeline: without an explicit clear the
                # marker from the last good frame stays on screen through every frame that
                # has no gaze, which reads as tracking that never drops out.
                rr.log(entity, rr.Clear.flat())

    rr.notebook_show()


print("Shared helpers defined.")

---

# Part A — On-device geometric ET

The geometric algorithm runs on the glasses in real time and writes its output into the VRS
`eyegaze` stream as the recording is made. No upload, no post-processing, no network — if you have
the VRS file, you have the gaze.

**Getting geometric ET data.** It is what `profile8` records. The ML algorithm in Part B needs
`profile8_ml_et` instead — a single recording carries one or the other, never both.

This part covers loading it, reading every field, converting gaze angles into 3D geometry, and
drawing the result on the RGB camera image.

In [ ]:
from projectaria_tools.core import data_provider

geometric_provider = data_provider.create_vrs_data_provider(geometric_vrs_file_path)
geometric_eyegaze_stream_id, geometric_num_samples = find_eyegaze_stream(
    geometric_provider, geometric_vrs_file_path
)

print(f"Opened {geometric_vrs_file_path}")
print(f"  eyegaze stream id: {geometric_eyegaze_stream_id}")
print(f"  eyegaze samples:   {geometric_num_samples}")

## A.1 Stream configuration

Every eyegaze stream carries a configuration record describing the algorithm that produced it:

```
vrs_data_provider.get_eye_gaze_configuration(stream_id)
```

**`EyeGazeConfiguration` fields**

| Field Name | Description |
| :-- | :-- |
| `stream_id` | Numeric id of the eyegaze stream |
| `algorithm_name` | Name of the on-device eye-tracking algorithm. Often empty — it is not a reliable way to tell the two on-device algorithms apart, see B.2 |
| `algorithm_version` | Algorithm version, formatted `"Major.minor"` |
| `nominal_rate_hz` | Nominal sample rate of the stream |
| `user_calibrated` | Whether an in-session user eye-tracking calibration was performed before the recording |
| `user_calibration_error` | Accuracy of that calibration — lower is better |
| `user_calibration_params_json` | Calibration parameter blob. Always empty for the geometric algorithm |
| `field_provenance_single` | Packed per-field provenance for the per-eye fields |
| `field_provenance_combined` | Packed per-field provenance for the combined fields |

Both on-device algorithms use the same stream label, so at some point you will want to know which
one produced a file. `algorithm_name` will not tell you — the field provenance table covered in
B.2 is what does.

In [ ]:
geometric_config = geometric_provider.get_eye_gaze_configuration(geometric_eyegaze_stream_id)

print("=== Geometric ET configuration ===")
print(f"  stream_id:                    {geometric_config.stream_id}")
print(f"  algorithm_name:               '{geometric_config.algorithm_name}'")
print(f"  algorithm_version:            '{geometric_config.algorithm_version}'")
print(f"  nominal_rate_hz:              {geometric_config.nominal_rate_hz}")
print(f"  user_calibrated:              {geometric_config.user_calibrated}")
print(f"  user_calibration_error:       {geometric_config.user_calibration_error}")
print(f"  user_calibration_params_json: {len(geometric_config.user_calibration_params_json)} chars")
print(f"  field_provenance_single:      0x{geometric_config.field_provenance_single:08x}")
print(f"  field_provenance_combined:    0x{geometric_config.field_provenance_combined:08x}")

## A.2 Reading `EyeGaze` samples

Eye gaze is queried like any other sensor stream, with the same APIs covered in
`Tutorial_1_vrs_data_provider_basics`:

- `vrs_data_provider.get_eye_gaze_data_by_index(stream_id, index)` — query by index
- `vrs_data_provider.get_eye_gaze_data_by_time_ns(stream_id, time_ns, time_domain, query_options)` — query by timestamp

**`EyeGaze` fields**

| Field Name | Description |
| :-- | :-- |
| `tracking_timestamp` | Timestamp of the eye tracking camera frame in the device time domain. A `datetime.timedelta`, not an integer |
| `session_uid` | Unique id for the calibration session. Changes at each in-session recalibration |
| `yaw`, `pitch` | Combined gaze angles in radians, in the CPF frame |
| `depth` | Depth in meters of the 3D gaze point in CPF. **`0` means unavailable, not zero meters** |
| `yaw_low`, `yaw_high`, `pitch_low`, `pitch_high` | Confidence interval bounds. The estimate lies inside the interval but is not necessarily centered |
| `combined_gaze_origin_in_cpf` | Combined gaze origin in CPF |
| `combined_gaze_valid` | Whether `combined_gaze_origin_in_cpf` is populated |
| `spatial_gaze_point_in_cpf` | 3D spatial gaze point in CPF |
| `spatial_gaze_point_valid` | Whether `spatial_gaze_point_in_cpf` is populated |
| `vergence` | Per-eye detail, see below |

**`vergence` (`EyeGazeVergence`) fields**

| Field Name | Description |
| :-- | :-- |
| `left_yaw`, `right_yaw` | Per-eye yaw in radians, in CPF |
| `left_pitch`, `right_pitch` | Per-eye pitch in radians, in CPF |
| `left_yaw_low/high`, `right_yaw_low/high` | Per-eye yaw confidence intervals |
| `tx_left_eye`, `ty_left_eye`, `tz_left_eye` | Left eye origin in CPF |
| `tx_right_eye`, `ty_right_eye`, `tz_right_eye` | Right eye origin in CPF |
| `left_blink`, `right_blink` | Per-eye blink detection |
| `left_gaze_valid`, `right_gaze_valid`, `left_blink_valid`, `right_blink_valid` | Per-eye validity flags |

:::note Pupil position and diameter are not exposed to Python
The underlying C++ `EyeGazeVergence` struct also carries entrance pupil position and pupil diameter
per eye, but those members are **not currently bound into the Python `EyeGazeVergence`**. They are
not readable from Python today.
:::

In [ ]:
# Query by index. geometric_num_samples came back from find_eyegaze_stream, which
# already guaranteed it is non-zero.
selected_index = min(5, geometric_num_samples - 1)
geometric_sample = geometric_provider.get_eye_gaze_data_by_index(
    geometric_eyegaze_stream_id, selected_index
)
print_eye_gaze_sample(geometric_sample, f"Geometric ET, sample #{selected_index} (by index)")

# Query by timestamp. Asking for the sample nearest the timestamp we just read back
# should return the very same sample.
query_time_ns = timestamp_to_ns(geometric_sample.tracking_timestamp)
sample_by_time = geometric_provider.get_eye_gaze_data_by_time_ns(
    geometric_eyegaze_stream_id,
    query_time_ns,
    TimeDomain.DEVICE_TIME,
    TimeQueryOptions.CLOSEST,
)
print(f"Queried at {query_time_ns} ns with TimeQueryOptions.CLOSEST")
print(f"  got sample at {timestamp_to_ns(sample_by_time.tracking_timestamp)} ns")
print(f"  same sample:  {timestamp_to_ns(sample_by_time.tracking_timestamp) == query_time_ns}")

## A.3 Gaze geometry helpers

`projectaria_tools.core.mps` ships six helpers for turning gaze angles into 3D geometry. They are
plain math on angles and work with an `EyeGaze` from any source, but they are **not
interchangeable** — three of them bake in a 63 mm inter-pupillary distance and assume both eyes
share a single pitch.

| Helper | Signature | Use when |
| :-- | :-- | :-- |
| `get_unit_vector_from_yaw_pitch` | `(yaw, pitch) -> vec3` | You have a combined gaze direction and want a unit ray |
| `get_eyegaze_point_at_depth` | `(yaw, pitch, depth) -> vec3` | You have a direction and a known depth. Returns zeros if `depth < 0` |
| `get_gaze_intersection_point` | `(left_yaw, right_yaw, pitch) -> vec3` | Triangulating from per-eye yaws. **Assumes 63 mm IPD and a shared pitch** |
| `compute_depth_and_combined_gaze_direction` | `(left_yaw, right_yaw, pitch) -> (depth_m, combined_yaw, pitch)` | You need a combined direction and a depth from per-eye yaws. **Same assumption.** The returned pitch is the input pitch, unchanged |
| `get_gaze_vectors` | `(left_yaw, right_yaw, pitch) -> (left_dir, right_dir)` | You want the two per-eye unit rays. **Same assumption** |
| `get_gaze_vergence_point` | `(left_origin, left_yaw, left_pitch, right_origin, right_yaw, right_pitch) -> (point, is_real)` | You have real per-eye origins and independent per-eye pitches. See Part B |

In [ ]:
from projectaria_tools.core.mps import (
    compute_depth_and_combined_gaze_direction,
    get_eyegaze_point_at_depth,
    get_gaze_intersection_point,
    get_gaze_vectors,
    get_unit_vector_from_yaw_pitch,
)

eye_gaze = geometric_sample
vergence = eye_gaze.vergence

print("=== Combined-direction helpers ===")
unit_vector = get_unit_vector_from_yaw_pitch(eye_gaze.yaw, eye_gaze.pitch)
print(f"  get_unit_vector_from_yaw_pitch -> {unit_vector}")

if eye_gaze.depth > 0:
    point_at_depth = get_eyegaze_point_at_depth(eye_gaze.yaw, eye_gaze.pitch, eye_gaze.depth)
    print(f"  get_eyegaze_point_at_depth     -> {point_at_depth}  (depth {eye_gaze.depth:.3f} m)")
else:
    print("  get_eyegaze_point_at_depth     -> skipped, depth is 0 (unavailable)")

print("\n=== Per-eye helpers (63 mm IPD and a shared pitch) ===")
intersection = get_gaze_intersection_point(vergence.left_yaw, vergence.right_yaw, eye_gaze.pitch)
print(f"  get_gaze_intersection_point    -> {intersection}")

depth_m, combined_yaw, combined_pitch = compute_depth_and_combined_gaze_direction(
    vergence.left_yaw, vergence.right_yaw, eye_gaze.pitch
)
print(f"  compute_depth_and_combined_gaze_direction ->")
print(f"      depth {depth_m:.4f} m, yaw {combined_yaw:.4f} rad, pitch {combined_pitch:.4f} rad")

left_direction, right_direction = get_gaze_vectors(
    vergence.left_yaw, vergence.right_yaw, eye_gaze.pitch
)
print(f"  get_gaze_vectors               -> left  {left_direction}")
print(f"                                    right {right_direction}")

## A.4 Coordinate frames, and drawing gaze on the RGB image

All eye tracking results in Aria are expressed in the **Central Pupil Frame (`CPF`)**, which sits
approximately at the center between the wearer's two eyes.

`CPF` is **not** the `Device` frame used by device calibration — the latter is the `slam-front-left`
camera on Gen 2 (`camera-slam-left` on Gen 1). Query the transform between them with:

```
device_calibration.get_transform_device_cpf()
```

So projecting a gaze point into a camera image is a two-step transform, `CPF -> Device -> Camera`,
followed by `camera_calib.project(...)`. That is exactly what the shared `gaze_point_to_pixel`
helper does, and what `overlay_gaze_on_rgb` calls once per frame below.

In [ ]:
geometric_device_calib = geometric_provider.get_device_calibration()
print(f"T_device_cpf translation (m): {geometric_device_calib.get_transform_device_cpf().translation()}")

print("\n=== Visualizing geometric ET gaze on RGB images ===")
overlay_gaze_on_rgb(
    provider=geometric_provider,
    gaze_sources=[
        (
            "geometric",
            lambda time_ns: geometric_provider.get_eye_gaze_data_by_time_ns(
                geometric_eyegaze_stream_id,
                time_ns,
                TimeDomain.DEVICE_TIME,
                TimeQueryOptions.CLOSEST,
            ),
            [255, 64, 255],  # magenta
        )
    ],
    rerun_app_name="rerun_viz_geometric_et",
)

---

# Part B — On-device ML ET

ML ET is the machine-learning eye-tracking algorithm. Like the geometric algorithm it runs on the
glasses in real time and writes into the same VRS `eyegaze` stream, but it delivers noticeably
better gaze quality.

**Getting ML ET data.** Which on-device algorithm runs is decided by the profile:

- `profile8_ml_et` — recording
- `profile9_ml_et` — streaming

Plain `profile8` records the geometric algorithm from Part A instead.

A recording made with one of these carries ML ET in its `eyegaze` stream. A single recording never
contains both algorithms.

This part is self-contained: it covers loading ML ET, its configuration, per-field provenance,
reading samples, the geometry helper that is correct for ML ET, and visualization.

In [ ]:
ml_et_provider = None
ml_et_eyegaze_stream_id = None
ml_et_num_samples = 0

if not ml_et_vrs_file_path:
    print("ml_et_vrs_file_path is not set - Part B will be skipped.")
else:
    ml_et_provider = data_provider.create_vrs_data_provider(ml_et_vrs_file_path)
    ml_et_eyegaze_stream_id, ml_et_num_samples = find_eyegaze_stream(
        ml_et_provider, ml_et_vrs_file_path
    )

    print(f"Opened {ml_et_vrs_file_path}")
    print(f"  eyegaze stream id: {ml_et_eyegaze_stream_id}")
    print(f"  eyegaze samples:   {ml_et_num_samples}")

## B.1 Stream configuration

Same API as Part A — `get_eye_gaze_configuration(stream_id)` — but two fields behave differently on
an ML ET recording:

- `algorithm_name` and `algorithm_version` name the algorithm that produced the stream. The writer
  passes both straight through for every source and does not guarantee either is populated, so
  `algorithm_name` comes back empty on plenty of genuine ML ET recordings. Do not reach for them
  to tell ML ET from geometric — B.2 has the field that actually settles it.
- `user_calibration_params_json` carries the in-session calibration parameters verbatim, but
  **only** when `user_calibrated` is `True`. The writer forces it empty in every other case,
  including for the geometric algorithm. A non-empty value therefore means "ML ET, with a
  calibration the wearer actually performed".

In [ ]:
if ml_et_provider is None:
    print("ml_et_vrs_file_path is not set - skipping.")
else:
    ml_et_config = ml_et_provider.get_eye_gaze_configuration(ml_et_eyegaze_stream_id)

    print("=== ML ET configuration ===")
    print(f"  stream_id:                    {ml_et_config.stream_id}")
    print(f"  algorithm_name:               '{ml_et_config.algorithm_name}'")
    print(f"  algorithm_version:            '{ml_et_config.algorithm_version}'")
    print(f"  nominal_rate_hz:              {ml_et_config.nominal_rate_hz}")
    print(f"  user_calibrated:              {ml_et_config.user_calibrated}")
    print(f"  user_calibration_error:       {ml_et_config.user_calibration_error}")
    print(f"  user_calibration_params_json: {len(ml_et_config.user_calibration_params_json)} chars")
    print(f"  field_provenance_single:      0x{ml_et_config.field_provenance_single:08x}")
    print(f"  field_provenance_combined:    0x{ml_et_config.field_provenance_combined:08x}")

## B.2 Field provenance

ML ET measures a subset of the `EyeGaze` fields directly and fills the rest in other ways. The
configuration record records, per field, how each value came to be:

| `FieldProvenance` | Meaning |
| :-- | :-- |
| `SUPPORTED` | Produced directly by the algorithm |
| `CALCULATED` | Derived on-device from other quantities |
| `HARDCODED` | Filled from a device constant. The `valid` flag reads `True`, but the value is not tracked |
| `NOT_PRODUCED` | Not produced; the corresponding `valid` flag is `False` |

Query it per field, with one enum for the per-eye fields and one for the combined fields:

- `config.get_single_field_provenance(SingleFieldId.X)`
- `config.get_combined_field_provenance(CombinedFieldId.X)`

**This is a property of the algorithm, not of your recording.** The writer stamps a fixed table onto
the configuration record according to which source produced the stream, so every ML ET recording
reports the same provenance. Two consequences follow, and both are worth knowing:

- The geometric source produces every field directly, so its table is left at the all-`SUPPORTED`
  default. **A table that reads all-`SUPPORTED` therefore means the stream is not ML ET.** That is
  the most useful thing this cell can tell you if `ml_et_vrs_file_path` is pointing at the wrong
  file — `algorithm_name` will not flag it for you, since it is written for both sources and is
  commonly empty on either. The cell below checks the table explicitly.
- On ML ET, `GAZE_ORIGIN` and `GAZE_ORIGIN_COMBINED` come back `HARDCODED`. The eye origins you read
  out of `vergence` are a device constant, not this wearer's anatomy. They are still the correct
  input to `get_gaze_vergence_point` in B.4, but do not read them as a measured inter-pupillary
  distance.

In [ ]:
from projectaria_tools.core.sensor_data import (
    CombinedFieldId,
    FieldProvenance,
    SingleFieldId,
)

SINGLE_FIELD_IDS = [
    SingleFieldId.GAZE_ORIGIN,
    SingleFieldId.GAZE_DIRECTION,
    SingleFieldId.ENTRANCE_PUPIL_POSITION,
    SingleFieldId.PUPIL_DIAMETER,
    SingleFieldId.BLINK,
]

COMBINED_FIELD_IDS = [
    CombinedFieldId.GAZE_ORIGIN_COMBINED,
    CombinedFieldId.GAZE_DIRECTION_COMBINED,
    CombinedFieldId.CONVERGENCE_DISTANCE,
    CombinedFieldId.INTEROCULAR_DISTANCE,
    CombinedFieldId.FOVEATED_GAZE,
    CombinedFieldId.SPATIAL_GAZE_POINT,
]

if ml_et_provider is None:
    print("ml_et_vrs_file_path is not set - skipping.")
else:
    # Keep the two tables separate. SingleFieldId and CombinedFieldId are both
    # IntEnums starting at 0, so merging them into one dict would silently
    # collapse entries that happen to share a numeric value.
    single_provenance = {
        field_id: ml_et_config.get_single_field_provenance(field_id)
        for field_id in SINGLE_FIELD_IDS
    }
    combined_provenance = {
        field_id: ml_et_config.get_combined_field_provenance(field_id)
        for field_id in COMBINED_FIELD_IDS
    }

    print("=== Field provenance ===")
    for field_id, provenance in single_provenance.items():
        print(f"  single/{field_id.name:<32} {provenance.name}")
    for field_id, provenance in combined_provenance.items():
        print(f"  combined/{field_id.name:<30} {provenance.name}")

    every_field = (*single_provenance.values(), *combined_provenance.values())
    if all(provenance == FieldProvenance.SUPPORTED for provenance in every_field):
        print("\n  Every field reads SUPPORTED. That is the default the writer leaves in")
        print("  place for a non-ML-ET stream, so this recording is almost certainly")
        print("  geometric ET rather than ML ET.")
        print(f"  algorithm_name is '{ml_et_config.algorithm_name}' - note that it is")
        print("  written for both sources, so a healthy-looking value here does not on")
        print("  its own confirm ML ET. Check the profile the recording was made with.")
    else:
        hardcoded = [
            f"single/{field_id.name}"
            for field_id, provenance in single_provenance.items()
            if provenance == FieldProvenance.HARDCODED
        ] + [
            f"combined/{field_id.name}"
            for field_id, provenance in combined_provenance.items()
            if provenance == FieldProvenance.HARDCODED
        ]
        print(f"\n  HARDCODED (valid reads True, but the value is a device constant):")
        for name in hardcoded:
            print(f"    {name}")

## B.3 Reading `EyeGaze` samples

Identical APIs to Part A — `get_eye_gaze_data_by_index` and `get_eye_gaze_data_by_time_ns` — and the
identical `EyeGaze` structure, so the field tables in section A.2 apply here unchanged.

Two things to keep in mind for ML ET specifically:

- `vergence.left_pitch` and `vergence.right_pitch` are estimated **independently per eye**. On Gen 1
  both eyes shared a single pitch. This matters for the geometry in the next section.
- Blink, entrance pupil position and pupil diameter are **not produced** by ML ET — see the
  provenance table in B.2. `left_blink_valid` and `right_blink_valid` read `False` accordingly, so
  the blink fields printed below carry no information on this source.

In [ ]:
if ml_et_provider is None:
    print("ml_et_vrs_file_path is not set - skipping.")
else:
    ml_et_index = min(5, ml_et_num_samples - 1)
    ml_et_sample = ml_et_provider.get_eye_gaze_data_by_index(
        ml_et_eyegaze_stream_id, ml_et_index
    )
    print_eye_gaze_sample(ml_et_sample, f"ML ET, sample #{ml_et_index} (by index)")

    ml_et_query_time_ns = timestamp_to_ns(ml_et_sample.tracking_timestamp)
    ml_et_by_time = ml_et_provider.get_eye_gaze_data_by_time_ns(
        ml_et_eyegaze_stream_id,
        ml_et_query_time_ns,
        TimeDomain.DEVICE_TIME,
        TimeQueryOptions.CLOSEST,
    )
    print(f"Queried at {ml_et_query_time_ns} ns with TimeQueryOptions.CLOSEST")
    print(f"  got sample at {timestamp_to_ns(ml_et_by_time.tracking_timestamp)} ns")

    print(f"\nPer-eye pitch difference in this sample: "
          f"{abs(ml_et_sample.vergence.left_pitch - ml_et_sample.vergence.right_pitch) * 1000:.3f} mrad")

## B.4 Gaze geometry: use `get_gaze_vergence_point`

Section A.3 listed six helpers. Three of them (`get_gaze_intersection_point`,
`compute_depth_and_combined_gaze_direction`, `get_gaze_vectors`) hardcode a 63 mm inter-pupillary
distance and assume both eyes share one pitch. Neither is right for Gen 2: the per-eye pitches are
estimated independently, and the eye origins come from `vergence` rather than from a constant baked
into the helper. (On ML ET those origins are themselves a device constant, per B.2 — what
`get_gaze_vergence_point` genuinely adds over the older helpers is the independent per-eye pitch.)

`get_gaze_vergence_point` is the one that takes those actual inputs:

```
get_gaze_vergence_point(left_origin, left_yaw, left_pitch,
                        right_origin, right_yaw, right_pitch) -> (point, is_real)
```

It builds one ray per eye and returns the midpoint of the shortest segment connecting them.

:::caution `is_real` is not a formality
The return value is a **pair**. `is_real` is `False` when the input geometry is degenerate, and in
that case the returned point is a **synthetic fallback placed 10 m straight ahead** — a
physically plausible forward point so that downstream consumers always get something, but not a
measurement. Do not plot it as data.

The guardrails that trigger the fallback are: vergence angle below 0.75°, rays that diverge, rays
whose closest approach misses by more than 3 cm, and convergence beyond roughly 6 m.
:::

In [ ]:
from projectaria_tools.core.mps import get_gaze_vergence_point

if ml_et_provider is None:
    print("ml_et_vrs_file_path is not set - skipping.")
else:
    ml_et_vergence = ml_et_sample.vergence

    left_origin = np.array(
        [ml_et_vergence.tx_left_eye, ml_et_vergence.ty_left_eye, ml_et_vergence.tz_left_eye]
    )
    right_origin = np.array(
        [ml_et_vergence.tx_right_eye, ml_et_vergence.ty_right_eye, ml_et_vergence.tz_right_eye]
    )

    vergence_point, is_real_vergence = get_gaze_vergence_point(
        left_origin,
        ml_et_vergence.left_yaw,
        ml_et_vergence.left_pitch,
        right_origin,
        ml_et_vergence.right_yaw,
        ml_et_vergence.right_pitch,
    )

    print("=== get_gaze_vergence_point ===")
    print(f"  left  eye origin (CPF): {left_origin}")
    print(f"  right eye origin (CPF): {right_origin}")
    print(f"  vergence point (CPF):   {vergence_point}")
    print(f"  is_real:                {is_real_vergence}")

    if not is_real_vergence:
        print("  Degenerate geometry: the point above is the 10 m forward fallback, not a measurement.")
    else:
        combined_origin = 0.5 * (left_origin + right_origin)
        print(f"  depth from combined origin: "
              f"{float(np.linalg.norm(vergence_point - combined_origin)):.4f} m")

    origin_separation_m = float(np.linalg.norm(left_origin - right_origin))
    print(f"\n  Eye origin separation in this stream: {origin_separation_m * 1000:.1f} mm")
    print("  GAZE_ORIGIN is HARDCODED on ML ET (see B.2), so this is a device constant,")
    print("  not a measurement of this wearer's inter-pupillary distance. Passing it to")
    print("  get_gaze_vergence_point is still correct - the helper's advantage over")
    print("  get_gaze_intersection_point and friends is the independent per-eye pitch,")
    print("  which is measured, rather than the origins, which are not.")

## B.5 Drawing ML ET gaze on the RGB image

The coordinate frames are the same as in Part A: gaze is in **CPF**, the cameras are calibrated in
**Device**, and `device_calibration.get_transform_device_cpf()` bridges the two. Projection is
`CPF -> Device -> Camera`, then `camera_calib.project(...)`.

In [ ]:
if ml_et_provider is None:
    print("ml_et_vrs_file_path is not set - skipping.")
else:
    print("=== Visualizing ML ET gaze on RGB images ===")
    overlay_gaze_on_rgb(
        provider=ml_et_provider,
        gaze_sources=[
            (
                "ml_et",
                lambda time_ns: ml_et_provider.get_eye_gaze_data_by_time_ns(
                    ml_et_eyegaze_stream_id,
                    time_ns,
                    TimeDomain.DEVICE_TIME,
                    TimeQueryOptions.CLOSEST,
                ),
                [64, 255, 255],  # cyan
            )
        ],
        rerun_app_name="rerun_viz_ml_et",
    )

---

# Part C — MPS ML ET

Machine Perception Services runs ML ET in the cloud, without the compute and power limits of the
glasses. Today it delivers the same gaze quality as the on-device ML ET in Part B — the cloud side
is updated more often than device firmware, so expect the two to diverge in later releases. The
cost is that you have to upload the recording and wait for processing to finish.

**What turns it on.** Unlike the two on-device sources, MPS ML ET is not selected by a recording
profile. MPS re-runs gaze from the **ET camera frames stored in the recording**, so what matters is
that the recording carries the `camera-et-left` / `camera-et-right` streams — not which gaze
algorithm ran on the glasses.

**Its rate follows the ET cameras.** MPS emits one gaze sample per ET camera frame, so the output
rate is the ET camera rate, not the 30 Hz of the on-device `eyegaze` stream. ET cameras default to
5 Hz, which is where a stock-profile recording lands. To go faster, raise `et_cameras.rate_hz` in a
custom profile; cameras top out at **90 Hz**. The catch is that the ET cameras are capped at 5 Hz
whenever on-device gaze is enabled, so a high-rate ET recording means giving up the on-device
`eyegaze` stream for that recording — you get MPS gaze only.

**Output layout.** MPS writes eye gaze into an `eye_gaze/` subfolder of the MPS output directory:

- `eye_gaze/general_eye_gaze.csv` — gaze for any wearer, no per-user calibration required
- `eye_gaze/personalized_eye_gaze.csv` — present only when the wearer completed an in-app eye
  tracking calibration
- `eye_gaze/summary.json`

Older MPS outputs name these `generalized_eye_gaze.csv` and `calibrated_eye_gaze.csv`. The reader
accepts both spellings, so there is nothing to rename.

The result is the same `EyeGaze` structure produced by the two on-device sources, so everything you
do with it downstream is identical.

## C.1 Two ways to load it

Both are shown below; pick whichever fits how you are walking the data.

| | `mps.read_eyegaze(path)` | `MpsDataProvider` |
| :-- | :-- | :-- |
| Returns | A plain `list[EyeGaze]` | An object answering timestamp queries |
| Finds the files for you | No, you build the path | Yes, via `MpsDataPathsProvider` |
| Best for | Running your own analysis over the whole time series | Walking a VRS file and fetching the gaze that lines up with each frame |

In [ ]:
from projectaria_tools.core import mps

general_eyegaze = []
personalized_eyegaze = []

if not mps_folder_path:
    print("mps_folder_path is not set - Part C will be skipped.")
else:
    general_eyegaze_path = os.path.join(mps_folder_path, "eye_gaze", "general_eye_gaze.csv")
    personalized_eyegaze_path = os.path.join(
        mps_folder_path, "eye_gaze", "personalized_eye_gaze.csv"
    )

    if os.path.exists(general_eyegaze_path):
        general_eyegaze = mps.read_eyegaze(general_eyegaze_path)
        print(f"read_eyegaze: {len(general_eyegaze)} general eye gaze samples")
    else:
        print(f"No general eye gaze CSV at {general_eyegaze_path}")

    if os.path.exists(personalized_eyegaze_path):
        personalized_eyegaze = mps.read_eyegaze(personalized_eyegaze_path)
        print(f"read_eyegaze: {len(personalized_eyegaze)} personalized eye gaze samples")
    else:
        print("No personalized eye gaze CSV - the wearer did not run an in-app calibration.")

    if general_eyegaze:
        print()
        print_eye_gaze_sample(
            general_eyegaze[len(general_eyegaze) // 2], "MPS general eye gaze"
        )

In [ ]:
mps_data_provider = None

if not mps_folder_path:
    print("mps_folder_path is not set - skipping.")
else:
    mps_data_paths = mps.MpsDataPathsProvider(mps_folder_path).get_data_paths()
    print("Resolved MPS eye gaze paths:")
    print(f"  general:      {mps_data_paths.eyegaze.general_eyegaze}")
    print(f"  personalized: {mps_data_paths.eyegaze.personalized_eyegaze}")
    print(f"  summary:      {mps_data_paths.eyegaze.summary}")

    mps_data_provider = mps.MpsDataProvider(mps_data_paths)
    print("\nAvailability:")
    print(f"  has_general_eyegaze():      {mps_data_provider.has_general_eyegaze()}")
    print(f"  has_personalized_eyegaze(): {mps_data_provider.has_personalized_eyegaze()}")
    print(f"  get_eyegaze_version():      {mps_data_provider.get_eyegaze_version()}")

    # Timestamp queries take a device-time timestamp in nanoseconds and, like the VRS
    # accessors, a TimeQueryOptions controlling the lookup.
    if mps_data_provider.has_general_eyegaze() and general_eyegaze:
        probe_time_ns = timestamp_to_ns(
            general_eyegaze[len(general_eyegaze) // 2].tracking_timestamp
        )
        queried = mps_data_provider.get_general_eyegaze(probe_time_ns, TimeQueryOptions.CLOSEST)
        print(f"\nget_general_eyegaze({probe_time_ns}, CLOSEST)")
        if queried is None:
            print("  returned None - no sample near that timestamp")
        else:
            print(f"  yaw {queried.yaw:.4f} rad, pitch {queried.pitch:.4f} rad, "
                  f"depth {queried.depth:.4f} m")

# The MPS output carries no imagery, so the overlay in C.4 and the reprojection
# numbers in C.2 both need the VRS this output was produced from. Opened once here
# and reused by both.
mps_vrs_provider = None
if mps_vrs_file_path:
    mps_vrs_provider = data_provider.create_vrs_data_provider(mps_vrs_file_path)
    print(f"\nOpened {mps_vrs_file_path} for the overlay")
else:
    print("\nmps_vrs_file_path is not set - the C.4 overlay will be skipped")


## C.2 General vs personalized

**General** eye gaze is produced for every MPS eye gaze request. It works for any wearer with no
setup, which is what you want for data collected across many people.

**Personalized** eye gaze is only produced when the wearer completed the in-app eye tracking
calibration before recording. It is fitted to that individual's eyes and is more accurate for them,
but it exists only for recordings where the calibration was actually done — so always check
`has_personalized_eyegaze()` before reaching for it, and have a fallback.

In [ ]:
if mps_data_provider is None:
    print("mps_folder_path is not set - skipping.")
elif not mps_data_provider.has_personalized_eyegaze():
    print("This MPS output has no personalized eye gaze; the wearer did not run the in-app")
    print("calibration. Use general eye gaze.")
elif not general_eyegaze:
    print("No general eye gaze loaded - nothing to compare against.")
else:
    print("Comparing general and personalized gaze at the same instants:")
    step = max(1, len(general_eyegaze) // 5)
    for sample in general_eyegaze[::step][:5]:
        time_ns = timestamp_to_ns(sample.tracking_timestamp)
        personalized = mps_data_provider.get_personalized_eyegaze(
            time_ns, TimeQueryOptions.CLOSEST
        )
        if personalized is None:
            continue
        yaw_delta_mrad = abs(personalized.yaw - sample.yaw) * 1000
        pitch_delta_mrad = abs(personalized.pitch - sample.pitch) * 1000
        print(f"  t={time_ns} ns   yaw delta {yaw_delta_mrad:6.2f} mrad   "
              f"pitch delta {pitch_delta_mrad:6.2f} mrad")

In [ ]:
# The same gap, measured where it actually bites: the image.
if (
    mps_data_provider is not None
    and mps_data_provider.has_personalized_eyegaze()
    and general_eyegaze
    and mps_vrs_provider is not None
):
    device_calib = mps_vrs_provider.get_device_calibration()
    T_device_cpf = device_calib.get_transform_device_cpf()
    rgb_calib = device_calib.get_camera_calib(RGB_CAMERA_LABEL)

    pixel_deltas = []
    for sample in general_eyegaze:
        time_ns = timestamp_to_ns(sample.tracking_timestamp)
        personalized = mps_data_provider.get_personalized_eyegaze(
            time_ns, TimeQueryOptions.CLOSEST
        )
        if personalized is None or not (
            sample.spatial_gaze_point_valid and personalized.spatial_gaze_point_valid
        ):
            continue
        general_pixel = gaze_point_to_pixel(
            sample.spatial_gaze_point_in_cpf, rgb_calib, T_device_cpf
        )
        personalized_pixel = gaze_point_to_pixel(
            personalized.spatial_gaze_point_in_cpf, rgb_calib, T_device_cpf
        )
        if general_pixel is not None and personalized_pixel is not None:
            pixel_deltas.append(
                float(np.linalg.norm(np.asarray(general_pixel) - np.asarray(personalized_pixel)))
            )

    if pixel_deltas:
        width = float(rgb_calib.get_image_size()[0])
        median_px = float(np.median(pixel_deltas))
        print("general vs personalized, reprojected into the RGB image:")
        print(f"  median {median_px:.0f} px, max {max(pixel_deltas):.0f} px, "
              f"on a {width:.0f} px wide frame")
        print(f"  = {median_px / width * 100:.1f}% of the frame width")
        print("\nBoth are projected by the identical chain, so this is not a transform")
        print("error - it is how much the per-user calibration moves the estimate.")
else:
    print("Need general and personalized gaze plus mps_vrs_provider to compare.")

## C.3 How the 3D geometry in a Gen 2 CSV is produced

Worth knowing before you consume `depth` or `spatial_gaze_point_in_cpf` from an MPS CSV.

The Gen 2 eye gaze CSV stores **per-eye rays only** — no combined 3D point. When the reader loads
one it back-fills the 3D fields:

- `combined_gaze_origin_in_cpf` — the midpoint of the two eye origins
- `spatial_gaze_point_in_cpf` — computed with `get_gaze_vergence_point` from the per-eye rays
- `depth` — the distance from the combined origin to that point

`spatial_gaze_point_valid` mirrors the `is_real` flag described in Part B. When the vergence is
degenerate, the reader deliberately leaves `depth` at `0` rather than reporting the 10 m fallback
distance as if it were a measurement.

The practical consequence: on Gen 2 MPS data, `depth == 0` and `spatial_gaze_point_valid == False`
travel together, and both mean *no reliable vergence at this instant* — not *the wearer is looking
at something zero meters away*.

## C.4 Drawing MPS gaze on the RGB image

Identical projection to Parts A and B: gaze is in **CPF**, cameras are calibrated in **Device**, and
`get_transform_device_cpf()` bridges them. The imagery comes from `mps_vrs_provider`, opened in C.1
from the VRS this MPS output was produced from — pointing that at a different recording would line
MPS gaze up against unrelated frames.

**Draw the personalized gaze when it exists.** C.2 measured the gap: on this recording the two
sources land about 200 px apart on a 2560 px wide frame, nearly all of it in pitch. Overlay general
on a recording that has personalized available and the marker sits a hand's width off whatever the
wearer was looking at — which reads as a broken projection even though the projection is fine. So
the cell below applies the same preference order `viewer_mps` uses: **personalized first, general as
the fallback.**

In [ ]:
if mps_data_provider is None or not (
    mps_data_provider.has_personalized_eyegaze() or mps_data_provider.has_general_eyegaze()
):
    print("No MPS eye gaze available - skipping the visualization.")
elif mps_vrs_provider is None:
    print("mps_vrs_file_path is not set - skipping the visualization.")
    print("Set it to the VRS file that this MPS output was produced from.")
else:
    # Same preference order as viewer_mps.
    if mps_data_provider.has_personalized_eyegaze():
        source_name = "personalized"
        gaze_at_time_ns = lambda time_ns: mps_data_provider.get_personalized_eyegaze(
            time_ns, TimeQueryOptions.CLOSEST
        )
    else:
        source_name = "general"
        gaze_at_time_ns = lambda time_ns: mps_data_provider.get_general_eyegaze(
            time_ns, TimeQueryOptions.CLOSEST
        )

    print(f"=== Visualizing MPS {source_name} eye gaze on RGB images ===")
    overlay_gaze_on_rgb(
        provider=mps_vrs_provider,
        gaze_sources=[(source_name, gaze_at_time_ns, [255, 255, 64])],  # yellow
        rerun_app_name="rerun_viz_mps_ml_et",
    )

---

# Summary

**Choosing a source**

| | On-device geometric ET | On-device ML ET | MPS ML ET |
| :-- | :-- | :-- | :-- |
| Where it runs | On the glasses | On the glasses | Cloud |
| Latency | Real time | Real time | Upload + cloud processing |
| Needs a network | No | No | Yes |
| Gaze quality | Baseline | Better | Same as on-device ML ET today |
| Selected by | `profile8` | `profile8_ml_et` / `profile9_ml_et` | ET camera frames being present |
| Rate | 30 Hz | 30 Hz | One sample per ET camera frame — 5 Hz by default, up to 90 Hz with a custom profile |
| Per-user calibration | — | Optional, in session | Optional, produces `personalized_eye_gaze.csv` |
| Loaded with | `get_eye_gaze_data_by_*` | `get_eye_gaze_data_by_*` | `mps.read_eyegaze` / `MpsDataProvider` |

**Things worth carrying away**

- All three sources return the same `EyeGaze` structure, so downstream geometry code is portable
  between them.
- Both on-device sources share the `eyegaze` stream label, and `algorithm_name` does not reliably
  tell them apart — it is written for both sources and is often empty. The field provenance table on
  the configuration record is what actually identifies ML ET.
- MPS ML ET and on-device ML ET currently produce the same gaze quality. MPS is updated more often
  than device firmware, so that may not stay true.
- MPS gaze arrives at the ET camera rate, not at 30 Hz. If you need it dense, raise
  `et_cameras.rate_hz` in a custom profile — at the cost of the on-device gaze stream.
- Eye gaze is in the **CPF** frame, which is not the **Device** frame. Use
  `get_transform_device_cpf()` before projecting into a camera.
- `depth == 0` means unavailable, not zero meters.
- When an MPS output has `personalized_eye_gaze.csv`, use it. On the sample recording it
  differs from general gaze by ~200 px once reprojected — large enough to look like a
  projection bug.
- On Gen 2, `get_gaze_vergence_point` is the geometry helper that uses the real eye origins and
  independent per-eye pitches; check its `is_real` return value before treating the point as data.

**Related tutorials**

- `Tutorial_1_vrs_data_provider_basics` — the query APIs used throughout
- `Tutorial_2_device_calibration` — the `Device` frame and camera projection
- `Tutorial_4_on_device_eyetracking_handtracking` — the other on-device machine perception stream
- `Tutorial_7_mps_data_provider_basics` — the rest of the MPS outputs